# Exploración — módulo de identificación de EMICRON 2024

Alcance: este notebook inspecciona únicamente `Módulo de identificación.csv`. No realiza cruces con otros módulos ni infiere categorías de negocio que no estén documentadas en este archivo.

## 1. Cargar el archivo

El archivo original se conserva sin modificaciones en `data/raw/`.

In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd().resolve().parent
FILE_PATH = PROJECT_DIR / 'data' / 'raw' / 'Módulo de identificación.csv'

df = pd.read_csv(FILE_PATH, encoding='utf-8-sig')
print(f'Archivo: {FILE_PATH.name}')
print(f'Tamaño: {df.shape[0]:,} filas × {df.shape[1]} columnas')

Archivo: Módulo de identificación.csv
Tamaño: 78,501 filas × 20 columnas


## 2. Estructura y completitud

In [3]:
print('Columnas:')
print(df.columns.tolist())

print('\nTipos de datos:')
display(df.dtypes.to_frame(name='dtype'))

print('\nPrimeras cinco filas:')
display(df.head())

print('\nInformación del dataframe:')
df.info()

missingness = pd.DataFrame({
    'valores_faltantes': df.isna().sum(),
    'porcentaje_faltante': (df.isna().mean() * 100).round(2)
}).sort_values('valores_faltantes', ascending=False)
print('\nValores faltantes:')
display(missingness)

Columnas:
['DIRECTORIO', 'SECUENCIA_P', 'SECUENCIA_ENCUESTA', 'COD_DEPTO', 'AREA', 'CLASE_TE', 'P35', 'P241', 'MES_REF', 'P3031', 'P3032_1', 'P3032_2', 'P3032_3', 'P3033', 'P3034', 'P3035', 'P3000', 'GRUPOS4', 'GRUPOS12', 'F_EXP']

Tipos de datos:


,dtype
DIRECTORIO,int64
SECUENCIA_P,int64
SECUENCIA_ENCUESTA,int64
COD_DEPTO,int64
AREA,float64
CLASE_TE,int64
P35,int64
P241,int64
MES_REF,str
P3031,int64



Primeras cinco filas:


,DIRECTORIO,SECUENCIA_P,SECUENCIA_ENCUESTA,COD_DEPTO,AREA,CLASE_TE,P35,P241,MES_REF,P3031,P3032_1,P3032_2,P3032_3,P3033,P3034,P3035,P3000,GRUPOS4,GRUPOS12,F_EXP
0,7627444,1,1,44,NaN,2,2,33,ENERO,2,NaN,NaN,NaN,2,240,2,2,2,3,60.050515
1,7627446,1,1,44,NaN,2,1,31,ENERO,2,NaN,NaN,NaN,2,166,2,2,1,1,86.341075
2,7627449,1,1,68,NaN,1,2,42,ENERO,2,NaN,NaN,NaN,2,60,2,2,3,5,139.884518
3,7627453,1,2,68,NaN,1,2,41,ENERO,2,NaN,NaN,NaN,2,60,2,2,4,7,168.635440
4,7627456,1,3,68,NaN,1,1,18,ENERO,2,NaN,NaN,NaN,2,9,2,2,4,6,64.659273



Información del dataframe:
<class 'pandas.DataFrame'>
RangeIndex: 78501 entries, 0 to 78500
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   DIRECTORIO          78501 non-null  int64  
 1   SECUENCIA_P         78501 non-null  int64  
 2   SECUENCIA_ENCUESTA  78501 non-null  int64  
 3   COD_DEPTO           78501 non-null  int64  
 4   AREA                52430 non-null  float64
 5   CLASE_TE            78501 non-null  int64  
 6   P35                 78501 non-null  int64  
 7   P241                78501 non-null  int64  
 8   MES_REF             78501 non-null  str    
 9   P3031               78501 non-null  int64  
 10  P3032_1             13225 non-null  float64
 11  P3032_2             13225 non-null  float64
 12  P3032_3             13225 non-null  float64
 13  P3033               78501 non-null  int64  
 14  P3034               78501 non-null  int64  
 15  P3035               78501 non-null  

,valores_faltantes,porcentaje_faltante
P3032_1,65276,83.15
P3032_2,65276,83.15
P3032_3,65276,83.15
AREA,26071,33.21
GRUPOS12,0,0.00
GRUPOS4,0,0.00
P3000,0,0.00
P3035,0,0.00
P3034,0,0.00
P3033,0,0.00


## 3. Unidad de análisis e identificadores

Se evalúa si la combinación de los tres identificadores técnicos identifica una fila de forma única.

In [4]:
KEY_COLUMNS = ['DIRECTORIO', 'SECUENCIA_P', 'SECUENCIA_ENCUESTA']
duplicate_key_rows = df.duplicated(KEY_COLUMNS, keep=False).sum()

key_summary = pd.DataFrame({
    'indicador': ['filas', 'claves únicas', 'filas con clave duplicada', 'filas completamente duplicadas'],
    'valor': [
        len(df),
        df[KEY_COLUMNS].drop_duplicates().shape[0],
        duplicate_key_rows,
        df.duplicated().sum(),
    ],
})
display(key_summary)

,indicador,valor
0,filas,78501
1,claves únicas,78501
2,filas con clave duplicada,0
3,filas completamente duplicadas,0


## 4. Variables categóricas

Se muestran frecuencias para las variables con hasta 30 valores distintos. Los códigos deben interpretarse con el diccionario oficial antes de asignarles etiquetas sustantivas.

In [5]:
categorical_columns = [
    column for column in df.columns
    if 1 < df[column].nunique(dropna=True) <= 30 and column not in KEY_COLUMNS
]

for column in categorical_columns:
    print(f'\n### {column} ({df[column].nunique(dropna=True)} valores distintos)')
    frequencies = (
        df[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name='casos')
    )
    frequencies['porcentaje'] = (frequencies['casos'] / len(df) * 100).round(2)
    display(frequencies)


### COD_DEPTO (25 valores distintos)


,COD_DEPTO,casos,porcentaje
0,52,5991,7.63
1,70,5333,6.79
2,13,4853,6.18
3,68,4560,5.81
4,5,4403,5.61
5,8,4234,5.39
6,76,4228,5.39
7,47,4217,5.37
8,23,4015,5.11
9,50,3148,4.01



### AREA (24 valores distintos)


,AREA,casos,porcentaje
0,NaN,26071,33.21
1,70.0,3764,4.79
2,13.0,3628,4.62
3,52.0,3295,4.20
4,8.0,3274,4.17
5,5.0,3135,3.99
6,68.0,2738,3.49
7,76.0,2725,3.47
8,23.0,2656,3.38
9,47.0,2269,2.89



### CLASE_TE (2 valores distintos)


...

,P3000,casos,porcentaje
0,2,68006,86.63
1,1,10495,13.37



### GRUPOS4 (5 valores distintos)


,GRUPOS4,casos,porcentaje
0,4,38756,49.37
1,3,20982,26.73
2,1,10710,13.64
3,2,8049,10.25
4,5,4,0.01



### GRUPOS12 (13 valores distintos)


,GRUPOS12,casos,porcentaje
0,5,20982,26.73
1,6,11142,14.19
2,1,10230,13.03
3,12,8625,10.99
4,3,8049,10.25
5,7,7523,9.58
6,4,4864,6.20
7,9,4451,5.67
8,11,946,1.21
9,10,798,1.02


## 5. Alcance para identificar tiendas

Esta exploración solo establece qué variables están presentes. Una categoría de tienda exige una variable documentada de actividad económica, tipo de comercio o código CIIU; no se debe inferir únicamente a partir de códigos sin diccionario.

In [6]:
df["GRUPOS12"].value_counts().sort_index()

GRUPOS12
1     10230
2       480
3      8049
4      4864
5     20982
6     11142
7      7523
8       407
9      4451
10      798
11      946
12     8625
13        4
Name: count, dtype: int64

In [7]:
df.groupby("GRUPOS12")["F_EXP"].agg(
    casos="size",
    poblacion_estimada="sum"
).sort_index()

,casos,poblacion_estimada
GRUPOS12,,
1,10230,1.138758e+06
2,480,5.634053e+04
3,8049,5.319877e+05
4,4864,3.006293e+05
5,20982,1.263754e+06
6,11142,6.288168e+05
7,7523,4.402232e+05
8,407,2.983981e+04
9,4451,2.769533e+05


In [8]:
resumen_grupos12 = (
    df.groupby("GRUPOS12")
      .agg(
          casos=("GRUPOS12", "size"),
          micronegocios_estimados=("F_EXP", "sum")
      )
      .reset_index()
)

resumen_grupos12

,GRUPOS12,casos,micronegocios_estimados
0,1,10230,1.138758e+06
1,2,480,5.634053e+04
2,3,8049,5.319877e+05
3,4,4864,3.006293e+05
4,5,20982,1.263754e+06
5,6,11142,6.288168e+05
6,7,7523,4.402232e+05
7,8,407,2.983981e+04
8,9,4451,2.769533e+05
9,10,798,4.586817e+04
